In [1]:
from typing import cast

from datasets import Dataset
from datasets import load_dataset as hf_load

mnist_test = cast(Dataset, hf_load("ylecun/mnist", split="test"))

/home/aweng/2033/dataeval-flow/.nox/docs/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from pathlib import Path

# Save to disk in HuggingFace arrow format
data_path = Path("./data/mnist/test")
mnist_test.save_to_disk(str(data_path))

Saving the dataset (0/1 shards):   0%|                                                                                                                                    | 0/10000 [00:00<?, ? examples/s]

Saving the dataset (1/1 shards): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 393957.13 examples/s]

Saving the dataset (1/1 shards): 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 10000/10000 [00:00<00:00, 380066.87 examples/s]

In [3]:
from dataeval_flow.config import (
    HuggingFaceDatasetConfig,
    PipelineConfig,
    SourceConfig,
)
from dataeval_flow.config.schemas import (
    DataSplittingTaskConfig,
    DataSplittingWorkflowConfig,
)
from dataeval_flow.workflow import run_task

workflow = DataSplittingWorkflowConfig(
    name="mnist_split",
    test_frac=0.2,  # 20% of full dataset held out for test
    val_frac=0.0,  # Must be 0 when num_folds > 1; validation is 1/num_folds
    num_folds=3,  # 3-fold cross-validation
    stratify=True,  # Preserve class distribution in each partition
)

task = DataSplittingTaskConfig(
    name="split_mnist",
    workflow="mnist_split",
    sources="mnist_src",
)

# Build the full pipeline config — datasets, sources, workflows, and tasks
config = PipelineConfig(
    datasets=[
        HuggingFaceDatasetConfig(name="mnist_test", path=str(data_path)),
    ],
    sources=[
        SourceConfig(name="mnist_src", dataset="mnist_test"),
    ],
    workflows=[workflow],
    tasks=[task],
)

In [4]:
result = run_task(task, config, cache_dir=Path("./cache"))

In [5]:
if not result.success:
    print(f"Workflow failed: {result.errors}")
assert result.success

In [6]:
print(result.report())


  DATASET SPLITTING: 10000 ITEMS → 3 FOLD(S)
  Timestamp:    2026-06-09T17:45:45.937326+00:00
  Duration:     2.51s
  Source:       mnist_src (mnist_test)
--------------------------------------------------------------------------------

  SUMMARY
  -------
  Class distribution (full dataset) ...................................   [..]
  Split sizes across folds .......................... 3 folds, test=2000  [..]
  Class distribution across splits (fold 0 of 3) ......................   [..]
  Stratification quality ...................... OK - max deviation 0.0pp  [ok]
  Pre-split balance (mutual information) ..............................   [..]
  Pre-split diversity .................................................   [..]

  Health: All checks passed [ok]

  CLASS DISTRIBUTION (FULL DATASET)
  Max/min class ratio: 1.3:1

  Class  Count
  -----  -----
  1       1135  ██████████████████████████████
  2       1032  ███████████████████████████▎
  7       1028  ███████████████████████████▏


In [7]:
raw = result.data.raw

print(f"Dataset size: {raw.dataset_size}")
print(f"Test indices: {len(raw.test_indices)}")
print(f"Number of folds: {len(raw.folds)}")

Dataset size: 10000
Test indices: 2000
Number of folds: 3


In [8]:
import polars as pl

rows = []
for i, fold in enumerate(raw.folds):
    rows.append({"fold": i, "train": len(fold.train_indices), "val": len(fold.val_indices)})
print(pl.DataFrame(rows))
print(f"\nTest (shared across folds): {len(raw.test_indices)} samples")

shape: (3, 3)
┌──────┬───────┬──────┐
│ fold ┆ train ┆ val  │
│ ---  ┆ ---   ┆ ---  │
│ i64  ┆ i64   ┆ i64  │
╞══════╪═══════╪══════╡
│ 0    ┆ 5333  ┆ 2667 │
│ 1    ┆ 5333  ┆ 2667 │
│ 2    ┆ 5334  ┆ 2666 │
└──────┴───────┴──────┘

Test (shared across folds): 2000 samples


In [9]:
# Verify no overlap between splits and full coverage per fold
test_set = set(raw.test_indices)

for i, fold in enumerate(raw.folds):
    train_set = set(fold.train_indices)
    val_set = set(fold.val_indices)

    assert train_set.isdisjoint(val_set), f"Fold {i}: train/val overlap!"
    assert train_set.isdisjoint(test_set), f"Fold {i}: train/test overlap!"
    assert val_set.isdisjoint(test_set), f"Fold {i}: val/test overlap!"

    total = len(train_set) + len(val_set) + len(test_set)
    assert total == raw.dataset_size, f"Fold {i}: missing indices: {total} != {raw.dataset_size}"

print(f"All {len(raw.folds)} folds verified: no overlap, full coverage.")

All 3 folds verified: no overlap, full coverage.


In [10]:
# Full dataset label stats
if raw.label_stats_full:
    print("Full dataset:")
    print(f"  Classes: {raw.label_stats_full.get('class_count', '?')}")
    print(f"  Per-class counts: {raw.label_stats_full.get('label_counts_per_class', [])}")

# Per-fold and test label stats
for i, fold in enumerate(raw.folds):
    if fold.label_stats_train:
        print(f"\nFold {i} train: {fold.label_stats_train.get('label_counts_per_class', [])}")
    if fold.label_stats_val:
        print(f"Fold {i} val:   {fold.label_stats_val.get('label_counts_per_class', [])}")
if raw.label_stats_test:
    print(f"\nTest:  {raw.label_stats_test.get('label_counts_per_class', [])}")

Full dataset:
  Classes: 10
  Per-class counts: {7: 1028, 2: 1032, 1: 1135, 0: 980, 4: 982, 9: 1009, 5: 892, 6: 958, 3: 1010, 8: 974}

Fold 0 train: {4: 524, 2: 550, 5: 476, 7: 548, 3: 539, 1: 606, 8: 519, 9: 538, 6: 511, 0: 522}
Fold 0 val:   {7: 274, 2: 276, 1: 302, 0: 262, 4: 262, 9: 269, 5: 237, 6: 256, 3: 269, 8: 260}

Fold 1 train: {7: 548, 2: 551, 1: 605, 0: 523, 4: 524, 9: 538, 5: 475, 6: 511, 3: 539, 8: 519}
Fold 1 val:   {4: 262, 2: 275, 5: 238, 7: 274, 3: 269, 1: 303, 8: 260, 9: 269, 6: 256, 0: 261}

Fold 2 train: {7: 548, 2: 551, 1: 605, 0: 523, 4: 524, 9: 538, 5: 475, 6: 512, 3: 538, 8: 520}
Fold 2 val:   {4: 262, 9: 269, 2: 275, 5: 238, 8: 259, 1: 303, 3: 270, 7: 274, 6: 255, 0: 261}

Test:  {4: 196, 5: 179, 2: 206, 3: 202, 1: 227, 7: 206, 6: 191, 8: 195, 9: 202, 0: 196}


In [11]:
# Pre-split balance — mutual information between factors and labels
balance_rows = raw.pre_split_balance.get("balance")
if balance_rows:
    print("Pre-split balance (mutual information):")
    print(pl.DataFrame(balance_rows))
else:
    print("No balance data (dataset may lack metadata factors)")

# Pre-split diversity — Shannon diversity per factor
diversity_rows = raw.pre_split_diversity.get("factors")
if diversity_rows:
    print("\nPre-split diversity:")
    print(pl.DataFrame(diversity_rows))
else:
    print("No diversity data (dataset may lack metadata factors)")

Pre-split balance (mutual information):
shape: (2, 2)
┌─────────────┬──────────┐
│ factor_name ┆ mi_value │
│ ---         ┆ ---      │
│ str         ┆ f64      │
╞═════════════╪══════════╡
│ class_label ┆ 1.0      │
│ id          ┆ 0.0      │
└─────────────┴──────────┘

Pre-split diversity:
shape: (2, 3)
┌─────────────┬─────────────────┬──────────────────┐
│ factor_name ┆ diversity_value ┆ is_low_diversity │
│ ---         ┆ ---             ┆ ---              │
│ str         ┆ f64             ┆ bool             │
╞═════════════╪═════════════════╪══════════════════╡
│ class_label ┆ 0.99612         ┆ false            │
│ id          ┆ 1.0             ┆ false            │
└─────────────┴─────────────────┴──────────────────┘


In [12]:
meta = result.metadata
print(f"Stratified:  {meta.stratified}")
print(f"Num folds:   {meta.num_folds}")
print(f"Split sizes: {meta.split_sizes}")

Stratified:  True
Num folds:   3
Split sizes: {'train': 5333, 'val': 2667, 'test': 2000}


In [13]:
import json

json_str = result.export(fmt="json")
exported = json.loads(json_str)

# Extract test indices (nested under "raw")
test_idx = exported["raw"]["test_indices"]
print(f"Test indices ({len(test_idx)} samples): {test_idx[:10]}...")

# Extract per-fold train/val indices
for i, fold in enumerate(exported["raw"]["folds"]):
    print(f"Fold {i}: train={len(fold['train_indices'])}, val={len(fold['val_indices'])}")

Test indices (2000 samples): [3685, 3700, 3718, 3722, 3726, 3728, 3752, 3758, 3770, 3780]...
Fold 0: train=5333, val=2667
Fold 1: train=5333, val=2667
Fold 2: train=5334, val=2666


In [14]:
from datasets import load_from_disk

ds = load_from_disk(str(data_path))

test_ds = ds.select(test_idx)
train_ds = ds.select(exported["raw"]["folds"][0]["train_indices"])
val_ds = ds.select(exported["raw"]["folds"][0]["val_indices"])

print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")

Train: 5333, Val: 2667, Test: 2000
